In [0]:
%run ../00-common/01.environment-config

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table = f'{catalog_name}.{gold_schema}.dim_races'

**Step 1 - Read source tables**
- `circuits`
- `races`

In [0]:
circuits_df = spark.read.table(f"{catalog_name}.{silver_schema}.circuits")
races_df = spark.read.table(f"{catalog_name}.{silver_schema}.races")

**Step 2 - Join `races` with `circuits` using `circuit_id`**

In [0]:
dim_races_df = (
            races_df
                .join(circuits_df, 
                    circuits_df.circuit_id == races_df.circuit_id,
                    "inner")
                .select(
                    races_df.season,
                    races_df.round,
                    races_df.race_name,
                    races_df.race_date,
                    circuits_df.circuit_name,
                    circuits_df.locality,
                    circuits_df.country
                )
)

In [0]:
display(dim_races_df)

**Step 3 - Write the transformed data to the `gold` `dim_races` table**

In [0]:
(
    dim_races_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(target_table)
)

In [0]:
%sql
select * from formula1.gold.dim_races